# Pre-entrenamiento con BERTAX para clasificación binaria

Instalar paquetes necesarios

In [ ]:
!pip install biopython
!pip install keras-pos-embd
!pip install keras_layer_normalization
!pip install keras_transformer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for keras-pos-embd: filename=keras_pos_embd-0.13.0-py3-none-any.whl size=6945 sha256=8d249a69023778939f23b21c4016454960c971e9cb575ad238a163b13fa437ba
  Stored in directory: /root/.cache/pip/wheels/f3/b3/f2/9acf9a5c6b16a27d3e9080901f49ae2774dcaca251a752d82a
Successfully built keras-pos-embd
  Preparing metadata (setup.py) ... done
  Created wheel for keras_layer_normalization: filename=keras_layer_normalization-0.16.0-py3-none-any.whl size=4654 sha256=c586105c808f4383f838ea0b8ff35df87d37703adf4891f6ebabf4e0a35270bd
  Stored in directory: /root/.cache/pip/wheels/0f/c4/a1/24f1ca7fd39e75f4d8dab7feda6fe3e2163d8062b29f1169fb
Successfully built keras_layer_normalization
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py

Importar las librerías necesarias

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import json
import os.path
from random import shuffle, sample
from tensorflow import keras
from sklearn.model_selection import train_test_split
from google.colab import drive
import sys
sys.path.append('/content/preprocessing')
from preprocessing.process_inputs import seq2kmers, ALPHABET
from preprocessing.generate_data import load_fragments
from model import PARAMS
from preprocessing.process_inputs import seq2kmers, ALPHABET
from dependencies.keras_bert.keras_bert.bert import get_model, compile_model
from dependencies.keras_bert.keras_bert.bert import gen_batch_inputs
from dependencies.keras_bert.keras_bert.bert import get_model, compile_model
from dependencies.keras_bert.keras_bert.bert import gen_batch_inputs
import argparse
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import Callback
from bert_utils import get_token_dict
from tensorflow import keras
from tensorflow.keras.optimizers import AdamW

Comprobar uso de GPUs

In [ ]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Num GPUs Available:  1


Declaración de clases:

**Clases**:

*   LossHistoryLogger
*   FragmentGenerator


In [ ]:
class LossHistoryLogger(Callback):
    def __init__(self, filename='loss_history.csv'):
        super().__init__()
        self.filename = filename
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs['epoch'] = epoch + 1
        self.history.append(logs)
        df = pd.DataFrame(self.history)
        df.to_csv(self.filename, index=False)

In [ ]:
class FragmentGenerator(Sequence):

    def __init__(self, fragments, species, seq_len, class_weights):
        self.fragments = fragments
        self.species = species
        self.seq_len = seq_len
        self.weight_classes = class_weights

    def get_sample_weight(self, taxid):
        # Clasificación binaria: cacao vs. nocacao
        if taxid == 3641:
            return self.weight_classes['binary'].get('Cacao', 1.0)
        else:
            return self.weight_classes['binary'].get('NotCacao', 1.0)

    def __len__(self):
        global batch_size
        return np.ceil(len(self.fragments)/float(batch_size)).astype(int)

    def __getitem__(self, idx):
        global token_dict, token_list
        batch_fragments = self.fragments[idx * batch_size:(idx+1) * batch_size]
        batch_species = self.species[idx * batch_size:(idx+1) * batch_size]
        batch_seqs = [seq2kmers(seq, k=3, stride=3, pad=False, to_upper=True)
                      for seq in batch_fragments]
        sentences = [[seq[:len(seq)//2], seq[len(seq)//2:]] for seq in batch_seqs]
        # Si el taxid es 3641, se asigna 1 ("Clasificada"); si no, 0 ("No clasificada")

        labels = [1 if int(taxid) == 3641 else 0 for taxid in batch_species]
        #return (gen_batch_inputs(sentences, token_dict, token_list, seq_len=self.seq_len)[0], np.array(labels))

        #return gen_batch_inputs(sentences, token_dict, token_list,
        #                        seq_len=self.seq_len)
        #
        #
        # Generar los inputs para el modelo (por ejemplo: input_ids, segment_ids, mask_ids, etc.)
        inputs_all = gen_batch_inputs(sentences, token_dict, token_list, seq_len=self.seq_len)
        # Generar etiquetas binarias: 1 para cacao (taxid 3641) y 0 para no cacao
        binary_labels = np.array([1 if taxid == 3641 else 0 for taxid in batch_species])
        # Calcular los sample weights para cada muestra
        sample_weights = np.array([self.get_sample_weight(taxid) for taxid in batch_species])
        # Retornar (inputs, targets, sample_weights)
        # Obtenemos el primer grupo de tensores (los que el modelo necesita)
        input_ids, segment_ids, input_mask = inputs_all[0]

        # Construimos la tupla para el modelo
        inputs_tuple = (input_ids, segment_ids, input_mask)

        # Usamos las etiquetas binarias y pesos de muestra que ya armaste
        return inputs_tuple, binary_labels, sample_weights

In [ ]:

#import tensorflow._api.v2.compat.v1 as tf
#tf.disable_v2_behavior()
#tf.compat.v1.disable_eager_execution()
#tf.config.run_functions_eagerly(True)
# Define manualmente los parámetros (ajusta las rutas y valores según necesites)
fragments_dir = "/content/27-3_input_bert_cn"  # Reemplaza con la ruta real
nr_seqs = 878208
seq_len = 151
batch_size = 64
val_split = 0.02
head_num = 4
transformer_num = 12
embed_dim = 256
feed_forward_dim = 2048
dropout_rate = 0.1
epochs = 10
no_balance = False
name = "bert_nc"

# Construcción del modelo
#from keras_bert import Tokenizer
token_dict = get_token_dict(ALPHABET, k=3)
token_list = list(token_dict)
#tokenizer = Tokenizer(token_dict) # Instantiate Tokenizer

# Si usas estrategia de distribución (GPU) y deseas usarla, puedes descomentar la siguiente línea:
# with mirrored_strategy.scope():
full_model = get_model(
    token_num=len(token_dict),
    head_num=head_num,
    transformer_num=transformer_num,
    embed_dim=embed_dim,
    feed_forward_dim=feed_forward_dim,
    seq_len=seq_len,
    pos_num=seq_len,
    dropout_rate=dropout_rate
)
model = keras.Model(
    inputs=full_model.input,
    outputs=full_model.get_layer('NSP').output
)
#compile_model(model)
#model.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-5), loss='sparse_categorical_crossentropy')#'binary_crossentropy'
model.compile(optimizer=AdamW(
                  #min_lr=0.0,
                  weight_decay=0.,
                  #decay_steps=100000,
                  #warmup_steps=10000,
                  learning_rate=5e-6,
                  #beta_1=0.9,
                  #beta_2=0.999,
                  #epsilon=1e-7,
                  #weight_decay_pattern=None,
                  #amsgrad=False
              ), loss='sparse_categorical_crossentropy')
model.summary()

# Definir las clases para la clasificación binaria
classes = ['Cacao', 'NotCacao']

# Cargar los datos de entrenamiento
fragments, y, species = load_fragments(
    fragments_dir, classes=classes,
    shuffle_=True, balance=(not no_balance),
    nr_seqs=nr_seqs
)

# Dividir en conjuntos de entrenamiento y validación
from sklearn.model_selection import train_test_split
f_train, f_val, s_train, s_val, c_train, c_val = train_test_split(
    fragments, species, y, test_size=val_split
)

# Calcular los pesos de muestra en el conjunto de entrenamiento
num_cacao = sum(1 for taxid in s_train if taxid == 3641)
num_no_cacao = len(s_train) - num_cacao
train_weights = {
    'binary': {
        'Cacao': 1 / (num_cacao / len(s_train)) if num_cacao > 0 else 1.0,
        'NotCacao': 1 / (num_no_cacao / len(s_train)) if num_no_cacao > 0 else 1.0
    }
}
num_cacao_val = sum(1 for taxid in s_val if taxid == 3641)
num_no_cacao_val = len(s_val) - num_cacao_val
val_weights = {
    'binary': {
        'Cacao': 1 / (num_cacao_val / len(s_val)) if num_cacao_val > 0 else 1.0,
        'NotCacao': 1 / (num_no_cacao_val / len(s_val)) if num_no_cacao_val > 0 else 1.0
    }
}

# Crear los generadores de datos para entrenamiento y validación
train_gen = FragmentGenerator(f_train, s_train, seq_len, train_weights)
print("Ana test:: train_gen ::")
print("Número de fragmentos:", len(train_gen.fragments))
print("Longitud de secuencia:", train_gen.seq_len)

val_gen = FragmentGenerator(f_val, s_val, seq_len, val_weights)

# Entrenar el modelo
model.fit(
    train_gen,
    epochs=epochs,
    validation_data=val_gen,
    batch_size=batch_size,
    validation_batch_size=batch_size,
    callbacks=[
        ModelCheckpoint(name + '_ep{epoch:02d}.keras'),
        EarlyStopping(
            monitor='val_loss',          # Métrica a vigilar
            patience=5,                  # Épocas sin mejora antes de detener
            restore_best_weights=True    # Recarga los mejores pesos automáticamente
        ),
        LossHistoryLogger(filename=name + '_loss_history.csv')
    ]
)

# Guardar el modelo final
model.save(name + '_trained.keras')


Exception reporting mode: Verbose
Ana test :: inputs: 
Tensor("Placeholder:0", shape=(None, 151, 256), dtype=float32)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Input-Token (InputLayer)  │ (None, 151)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Input-Segment             │ (None, 151)            │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Embedding-Token           │ (None, 151, 256)       │         17,664 │ Input-Token[0][0]      │
│ (TokenEmbedding)          │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Embedding-Segment         │ (None, 151, 256)       │            512 │ Input-Segment[0][0]    │
│ (Embedding)               │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add (Add)                 │ (None, 151, 256)       │              0 │ Embedding-Token[0][0], │
│                           │                        │                │ Embedding-Segment[0][… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Embedding-Position        │ (None, 151, 256)       │         38,656 │ add[0][0]              │
│ (PositionEmbedding)       │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Embedding-Dropout         │ (None, 151, 256)       │              0 │ Embedding-Position[0]… │
│ (Dropout)                 │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Embedding-Norm            │ (None, 151, 256)       │            512 │ Embedding-Dropout[0][… │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-MultiHeadSelfA… │ (None, 151, 256)       │        263,168 │ Embedding-Norm[0][0]   │
│ (MultiHeadAttention)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-MultiHeadSelfA… │ (None, 151, 256)       │              0 │ Encoder-1-MultiHeadSe… │
│ (Dropout)                 │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-MultiHeadSelfA… │ (None, 151, 256)       │              0 │ Embedding-Norm[0][0],  │
│ (Add)                     │                        │                │ Encoder-1-MultiHeadSe… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-MultiHeadSelfA… │ (None, 151, 256)       │            512 │ Encoder-1-MultiHeadSe… │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-FeedForward     │ (None, 151, 256)       │      1,050,880 │ Encoder-1-MultiHeadSe… │
│ (FeedForward)             │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-FeedForward

 Total params: 15,904,514 (60.67 MB)

 Trainable params: 15,904,514 (60.67 MB)

 Non-trainable params: 0 (0.00 B)

78160 fragments loaded in total; balanced=True, shuffle_=True, nr_seqs=39080
Ana test:: train_gen ::
Número de fragmentos: 76596
Longitud de secuencia: 151
Epoch 1/10
1197/1197 ━━━━━━━━━━━━━━━━━━━━ 281s 170ms/step - loss: 1.2289 - val_loss: 0.7043 - epoch: 1.0000
Epoch 2/10
1197/1197 ━━━━━━━━━━━━━━━━━━━━ 152s 127ms/step - loss: 0.7421 - val_loss: 0.6452 - epoch: 2.0000
Epoch 3/10
1197/1197 ━━━━━━━━━━━━━━━━━━━━ 151s 126ms/step - loss: 0.7051 - val_loss: 0.6225 - epoch: 3.0000
Epoch 4/10
1197/1197 ━━━━━━━━━━━━━━━━━━━━ 151s 126ms/step - loss: 0.6707 - val_loss: 0.6445 - epoch: 4.0000
Epoch 5/10
1197/1197 ━━━━━━━━━━━━━━━━━━━━ 151s 126ms/step - loss: 0.6758 - val_loss: 0.6233 - epoch: 5.0000
Epoch 6/10
1197/1197 ━━━━━━━━━━━━━━━━━━━━ 151s 126ms/step - loss: 0.6509 - val_loss: 0.6401 - epoch: 6.0000
Epoch 7/10
1197/1197 ━━━━━━━━━━━━━━━━━━━━ 151s 126ms/step - loss: 0.6539 - val_loss: 0.6425 - epoch: 7.0000
Epoch 8/10
1197/1197 ━━━━━━━━━━━━━━━━━━━━ 151s 126ms/step - loss: 0.6612 - val_loss: 0.6